In [ ]:
from bs4 import BeautifulSoup
import datetime
import pandas as pd
from pandas import ExcelWriter
from selenium import webdriver
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
import bvdpdf
from time import sleep
import os

print("CF CEMACCF Web Scraping Tool v.1.0")

#Assigning current time, output file name and ExcelWriter object
now = datetime.datetime.now()
filename = 'CF CEMACCF SQL Ready {}.xlsx'.format(str(now).replace(":",".")[:-7])
writer = ExcelWriter(filename)

#Assigning the folders that are going to be used in the process
scriptfolder = os.path.dirname(os.path.abspath(__file__))
tempfolder = os.path.join(scriptfolder,'tempfolder')
os.chdir(scriptfolder)

#Creating tempfolder if it doesn't exists, emptying in if it does exist
if os.path.exists(tempfolder):
	for temp_file in os.listdir(tempfolder):
		os.remove(os.path.join(tempfolder, temp_file))
else:
	os.mkdir(tempfolder)

#Starting Chrome driver, set to download files in tempfolder
chromeOptions = webdriver.ChromeOptions()
prefs = {"plugins.always_open_pdf_externally": True,
		 "download.prompt_for_download": False,
		 "download.default_directory" : tempfolder}
chromeOptions.add_experimental_option("prefs",prefs)
driver = webdriver.Chrome(options=chromeOptions)
driver.maximize_window(ChromeDriverManager().install())

#Creating dictionary with Regcodes and their respective URLs
regdict =   {'CF CEMACCF 2': 'https://www.beac.int/supervision-bancaire/lexique-etablissements-financiers/', 
             'CF CEMACCF 3': 'Tchad', 'CF CEMACCF 4': 'Guinée Equatoriale',
             'CF CEMACCF 5': 'Gabon', 'CF CEMACCF 6': 'Congo',
             'CF CEMACCF 7': 'Centra', 'CF CEMACCF 8': 'Cameroun',} #
             
other_regulators = {'CG': 'CEMACCG', 
                    'CM': 'CEMACCM', 
                    'GA': 'CEMACGA', 
                    'GQ': 'CEMACGQ', 
                    'TD': 'CEMACTD' }
            
ISO_country = {'Tchad': 'TD', 'Guinée Equatoriale' : 'GQ', 'Gabon' : 'GA', 'Congo': 'CD', 'Centra': 'CF', 'Cameroun' : 'CM'}

#Creating dictionary to containg regulators data and then be converted to a pandas' DataFrame
sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
		  'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
		  'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
		  'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
		  'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
		  'Phone - Mother company': [], 'Check': []}

processdate=now.strftime('%Y-%m-%d')

for reg in regdict:
    print('Working with {}'.format(reg))
    if ' 2' in reg:
        driver.get(regdict[reg])
        sleep(3)
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        table = soup.find('table', {'id': 'documents_contenu_cpt'})
        trs = table.find('tbody').find_all('tr')
        for tr in trs:
            tds = tr.find_all('td')
            sqldict['Name'].append(tds[0].text.strip())
            sqldict['Cntry'].append(ISO_country[tds[2].text.strip()])
            sqldict['RegCtry'].append('CF')
            sqldict['RegCode'].append('CEMACCF')
            sqldict['ListCode'].append(reg.split()[-1])
            sqldict['ListProcessDate'].append(processdate)
            for key in sqldict:
                if len(sqldict['Name']) > len(sqldict[key]):
                    sqldict[key].append('')
    else:
        driver.get('https://www.beac.int/supervision-bancaire/lexique-banques-de-cemac/')
        sleep(3)
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        pdf_table = soup.find('table', {'id': 'documents_contenu_cpt'})
        pdf_anchors = [ele for ele in pdf_table.find_all('a', href=True) if regdict[reg] in ele.text]
        href_pdf = [ele['href'] for ele in pdf_anchors if regdict[reg] in ele.text][0] #if index error: no matches found
        driver.get(href_pdf)
        for times in range(20):
            tempfiles = os.listdir(tempfolder)
            if len(tempfiles) > 0 and '.tmp' not in ''.join(tempfiles).lower() and '.crdownload' not in ''.join(tempfiles).lower():
                break
            sleep(2)
        else:
            dl_error = f'{reg} - Failed to download file - URL: {href_pdf}'
            raise Exception(dl_error)
        pdf_path = os.path.join(tempfolder, tempfiles[0])
        tables = bvdpdf.get_tables(pdf_path)
        os.remove(pdf_path)
        for i,table in enumerate(tables):
            if  table.shape[1]<3:#we skip tables with less than 3 columns
                continue
            if i > 0:#we need to take headers as first data row i in the second; third... tables
                table = table.columns.to_frame().T.append(table, ignore_index=True)
                table.columns = range(len(table.columns))
            table = table[table.iloc[:,1]!=''] #we only take the rows when there is a acronym or short company name in the correspondent column
            table = table.reset_index(drop=True)
            bank_names = table.iloc[:,0].tolist()
            bank_names = [b_n.split('DG',1)[0].strip() if 'DG:' in b_n.replace(' ', '') or 'DGA:' in b_n.replace(' ', '') else b_n for b_n in bank_names ]
            bank_names = [b_n for b_n in bank_names if len(b_n.strip()) > 0]
            sqldict['Name'].extend(bank_names)
            sqldict['Cntry'].extend([ISO_country[regdict[reg]] for ele in range(len(bank_names))])
            sqldict['RegCtry'].extend(['CF' for ele in range(len(bank_names))])
            sqldict['RegCode'].extend(['CEMACCF' for ele in range(len(bank_names))])
            sqldict['ListCode'].extend([reg.split()[-1] for ele in range(len(bank_names))])
            sqldict['ListProcessDate'].extend([processdate for ele in range(len(bank_names))])
            for key in sqldict:
                if len(sqldict['Name']) > len(sqldict[key]):
                    sqldict[key].extend(['' for ele in range(len(bank_names))])

    
os.chdir(scriptfolder)
df=pd.DataFrame(sqldict)

for regctry, regcode in other_regulators.items():
    df_other = df[df['Cntry'] == regctry]
    df_other['RegCode'] = regcode
    df_other['RegCtry'] = regctry
    df = pd.concat([df, df_other])

df.to_excel(writer, 'SQL Ready', index=False)

writer.save()
writer.close()
driver.quit()

#Moving the file to the output folder (this way it will be displayed in the Control Room)
sleep(3)

    
    